In [ ]:
#pip install folium

In [7]:
import pandas as pd
import folium

df = pd.read_csv("../data/raw_data/restaurants_raw_multiarea.csv")

AREAS = {
    "Baneshwor": (27.69396, 85.33738),
    "New Road":  (27.70200, 85.30743),
    "Koteshwor": (27.68333, 85.35000),
    "Bhaktapur durbar square": (27.67203, 85.42811),
    "Patan durbar square":     (27.67340, 85.32500),
    "Boudha stupa": (27.72139, 85.36194),
    "Pulchowk": (27.6787, 85.3175),
    "Durbar Marg": (27.71261, 85.31797),
    "Kirtipur": (27.67806, 85.27694),
}

COLORS = {
    "Baneshwor": "blue",
    "New Road": "green",
    "Koteshwor": "orange",
    "Bhaktapur durbar square": "purple",
    "Patan durbar square": "darkred",
    "Boudha stupa": "cadetblue",
    "Pulchowk": "darkgreen",
    "Durbar Marg": "pink",
    "Kirtipur": "black",
}

RADIUS = 1500  # meters, same radius used during collection

# fit the map to bounds that cover every search circle, with padding for the circles themselves
lats = [lat for lat, lng in AREAS.values()]
lngs = [lng for lat, lng in AREAS.values()]
pad = 0.02  # degrees
bounds = [[min(lats) - pad, min(lngs) - pad], [max(lats) + pad, max(lngs) + pad]]

m = folium.Map()
m.fit_bounds(bounds)

# draw each search circle
for area, center in AREAS.items():
    folium.Circle(
        center, radius=RADIUS, color=COLORS[area],
        fill=True, fill_opacity=0.08, popup=area
    ).add_to(m)

# plot each restaurant, coloured by its search area
for _, r in df.iterrows():
    folium.CircleMarker(
        location=(r["latitude"], r["longitude"]),
        radius=3,
        color=COLORS.get(r["search_area"], "gray"),
        fill=True, fill_opacity=0.8,
        popup=f"{r['restaurant_name']} ({r['search_area']})"
    ).add_to(m)

m.save("../outputs/multiarea_map.html")
print("Saved -> outputs/multiarea_map.html")
m

Saved -> outputs/multiarea_map.html


## Final dataset

### Load restaurants and POI data

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

restaurants = pd.read_csv("../data/raw_data/restaurants_raw_multiarea.csv")
pois = pd.read_csv("../data/processed/pois_unique.csv")

print(f"restaurants: {restaurants.shape}, pois: {pois.shape}")
print("duplicate restaurant place_ids:", restaurants["place_id"].duplicated().sum())
print("missing restaurant_rating / user_rating_count:",
      restaurants[["restaurant_rating", "user_rating_count"]].isna().sum().to_dict())
pois["poi_type"].value_counts()

restaurants: (1472, 13), pois: (13809, 8)
duplicate restaurant place_ids: 0
missing restaurant_rating / user_rating_count: {'restaurant_rating': 52, 'user_rating_count': 52}


poi_type
retail           3812
office           2451
clinic           1302
school           1201
temple           1139
bank             1094
recreation       1022
college           713
hospital          485
parking_space     251
museum            160
bus_stop          143
cinema             36
Name: count, dtype: int64

### POI counts within 500m of each restaurant

One `BallTree` (haversine metric) per POI category, so the radius query is vectorized across all restaurants at once instead of looping pairwise.

In [2]:
import pandas as pd
d = pd.read_csv("../data/processed/pois_unique.csv")
print(sorted(d["poi_type"].unique()))

['bank', 'bus_stop', 'cinema', 'clinic', 'college', 'hospital', 'museum', 'office', 'parking_space', 'recreation', 'retail', 'school', 'temple']


In [3]:
EARTH_RADIUS_M = 6_371_000
RADIUS_M = 500
radius_rad = RADIUS_M / EARTH_RADIUS_M

restaurant_coords_rad = np.radians(restaurants[["latitude", "longitude"]].values)

POI_CATEGORIES = ['bank', 'bus_stop', 'cinema', 'clinic', 'college', 'hospital', 'museum', 'office', 'parking_space', 'recreation', 'retail', 'school', 'temple']

poi_features = pd.DataFrame(index=restaurants.index)
for category in POI_CATEGORIES:
    subset = pois.loc[pois["poi_type"] == category, ["latitude", "longitude"]]
    col = f"{category}_count_500m"
    if subset.empty:
        poi_features[col] = 0
        continue
    tree = BallTree(np.radians(subset.values), metric="haversine")
    poi_features[col] = tree.query_radius(restaurant_coords_rad, r=radius_rad, count_only=True)

poi_features.describe()

,bank_count_500m,bus_stop_count_500m,cinema_count_500m,clinic_count_500m,college_count_500m,hospital_count_500m,museum_count_500m,office_count_500m,parking_space_count_500m,recreation_count_500m,retail_count_500m,school_count_500m,temple_count_500m
count,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000,1472.000000
mean,22.777853,2.120924,0.855978,24.622283,11.841712,7.742527,5.704484,39.479620,5.622283,17.902853,61.987772,19.095109,22.931386
std,20.508730,2.243879,1.169946,16.822624,11.782879,5.221889,8.266567,27.695236,5.443361,15.288459,33.155667,11.015553,25.338552
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,6.000000,0.000000,0.000000,12.000000,3.000000,4.000000,0.000000,15.000000,1.000000,11.000000,45.000000,12.000000,5.000000
50%,17.000000,1.000000,0.000000,23.000000,8.000000,8.000000,2.000000,37.000000,4.000000,16.000000,63.000000,19.000000,13.000000
75%,37.000000,3.000000,2.000000,37.000000,17.000000,11.000000,9.000000,66.000000,10.000000,20.000000,81.250000,27.000000,32.000000
max,95.000000,11.000000,7.000000,71.000000,62.000000,30.000000,36.000000,103.000000,23.000000,76.000000,156.000000,56.000000,104.000000


### Restaurant competition features

For each restaurant: `competitor_count_500m` and the mean `restaurant_rating` / `user_rating_count` of every *other* restaurant within 500m (`NaN` when none are found), plus `nearest_restaurant_m` — the true nearest-neighbour distance to the closest other restaurant, not limited to the 500m radius.

In [4]:
restaurant_tree = BallTree(restaurant_coords_rad, metric="haversine")

neighbor_idx = restaurant_tree.query_radius(restaurant_coords_rad, r=radius_rad)
ratings = restaurants["restaurant_rating"].to_numpy()
review_counts = restaurants["user_rating_count"].to_numpy()

n = len(restaurants)
competitor_count_500m = np.zeros(n, dtype=int)
avg_restaurant_rating = np.full(n, np.nan)
avg_review_ratings = np.full(n, np.nan)

for i, neighbors in enumerate(neighbor_idx):
    others = neighbors[neighbors != i]
    competitor_count_500m[i] = len(others)
    if len(others):
        avg_restaurant_rating[i] = np.nanmean(ratings[others])
        avg_review_ratings[i] = np.nanmean(review_counts[others])

# k=2: the nearest match to any point is itself (distance 0), so the 2nd column is the nearest *other* restaurant
dist_rad, _ = restaurant_tree.query(restaurant_coords_rad, k=2)
nearest_restaurant_m = dist_rad[:, 1] * EARTH_RADIUS_M

competition_features = pd.DataFrame({
    "competitor_count_500m": competitor_count_500m,
    "avg_restaurant_rating_500m": avg_restaurant_rating,
    "avg_review_ratings_500m": avg_review_ratings,
    "nearest_restaurant_m": nearest_restaurant_m,
}, index=restaurants.index)

competition_features.describe()

C:\Users\mausa\AppData\Local\Temp\ipykernel_26556\2760515524.py:16: RuntimeWarning: Mean of empty slice
  avg_restaurant_rating[i] = np.nanmean(ratings[others])
C:\Users\mausa\AppData\Local\Temp\ipykernel_26556\2760515524.py:17: RuntimeWarning: Mean of empty slice
  avg_review_ratings[i] = np.nanmean(review_counts[others])


,competitor_count_500m,avg_restaurant_rating_500m,avg_review_ratings_500m,nearest_restaurant_m
count,1472.000000,1432.000000,1432.000000,1472.000000
mean,40.828804,4.416182,151.016679,100.009680
std,34.428626,0.195568,103.766305,214.922051
min,0.000000,1.000000,1.000000,0.000000
25%,11.000000,4.317803,68.812500,18.443234
50%,31.000000,4.416905,136.675347,42.725668
75%,61.000000,4.522826,217.055556,106.203259
max,123.000000,5.000000,1471.000000,4327.573897


### Assemble & save final dataset

In [5]:
final_df = pd.concat([restaurants, poi_features, competition_features], axis=1)

out_path = "../data/processed/final_dataset.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved {final_df.shape[0]} rows x {final_df.shape[1]} columns -> {out_path}")
final_df.head()

Saved 1472 rows x 30 columns -> ../data/processed/final_dataset.csv


,place_id,restaurant_name,address,latitude,longitude,restaurant_rating,user_rating_count,price_level,primary_type,all_types,...,office_count_500m,parking_space_count_500m,recreation_count_500m,retail_count_500m,school_count_500m,temple_count_500m,competitor_count_500m,avg_restaurant_rating_500m,avg_review_ratings_500m,nearest_restaurant_m
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,"Pipal Bot Marg, Kathmandu 56900, Nepal",27.699546,85.337687,4.3,443.0,PRICE_LEVEL_MODERATE,restaurant,restaurant|food|point_of_interest|establishment,...,53,5,21,52,35,15,20,4.265000,87.700000,66.240461
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.4,1180.0,PRICE_LEVEL_EXPENSIVE,restaurant,restaurant|food|point_of_interest|establishment,...,66,6,16,80,39,13,44,4.247727,159.477273,3.882433
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,"4 Chhakku Bakku Marg, Kathmandu 44600, Nepal",27.689064,85.334295,4.8,90.0,NaN,restaurant,restaurant|food|point_of_interest|establishment,...,67,9,15,65,28,6,47,4.234043,204.255319,35.077195
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,"Janata Sadak, Kathmandu 44600, Nepal",27.681549,85.341340,4.9,60.0,NaN,restaurant,restaurant|food|point_of_interest|establishment,...,66,4,26,77,16,17,14,4.371429,83.214286,5.223270
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,"Shree Krishna Sadan, Chhakku Bakku Marg, Kathm...",27.688812,85.334080,4.7,43.0,NaN,restaurant,restaurant|food|point_of_interest|establishment,...,65,10,15,64,27,7,47,4.270213,200.893617,35.077195


In [6]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1472 entries, 0 to 1471
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   place_id                    1472 non-null   object 
 1   restaurant_name             1472 non-null   object 
 2   address                     1472 non-null   object 
 3   latitude                    1472 non-null   float64
 4   longitude                   1472 non-null   float64
 5   restaurant_rating           1420 non-null   float64
 6   user_rating_count           1420 non-null   float64
 7   price_level                 168 non-null    object 
 8   primary_type                1466 non-null   object 
 9   all_types                   1472 non-null   object 
 10  search_area                 1472 non-null   object 
 11  searched_as                 1472 non-null   object 
 12  reviews                     1366 non-null   object 
 13  bank_count_500m             1472 